In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 16

data = np.load("/kaggle/input/datasets/raedmuhammed/datasetready1/datasets (1).npz")

X_train_seq = data["X_train_seq"]
y_train = data["y_train"]

X_val_seq = data["X_val_seq"]
y_val = data["y_val"]

X_test_seq = data["X_test_seq"]
y_test = data["y_test"]

In [2]:
print(X_train_seq.shape)
print(y_train.shape)

print(X_val_seq.shape)
print(y_val.shape)

print(X_test_seq.shape)
print(y_test.shape)

(85715, 50)
(85715, 6)
(18367, 50)
(18367, 6)
(18368, 50)
(18368, 6)


In [3]:

train_dataset = (
    tf.data.Dataset.from_tensor_slices((X_train_seq, y_train))
    .cache()
    .shuffle(len(X_train_seq), seed=42, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


val_dataset = (
    tf.data.Dataset.from_tensor_slices((X_val_seq, y_val))
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


test_dataset = (
    tf.data.Dataset.from_tensor_slices((X_test_seq, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("train batches:", len(train_dataset))
print("val batches:  ", len(val_dataset))
print("test batches: ", len(test_dataset))

I0000 00:00:1790988550.804817      59 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790988550.808503      59 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


train batches: 5358
val batches:   1148
test batches:  1148


In [4]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_seq, y_train))
train_dataset = train_dataset.cache().shuffle(len(X_train_seq)).batch(16).prefetch(8)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val_seq, y_val))
val_dataset = val_dataset.cache().batch(16).prefetch(8)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test_seq, y_test))
test_dataset = test_dataset.cache().batch(16).prefetch(8)

In [5]:
pos_counts = y_train.sum(axis=0)
neg_counts = y_train.shape[0] - pos_counts

# define class weights 
class_weights = np.log1p(neg_counts / pos_counts) 

def weighted_binary_crossentropy(class_weights):
    class_weights = tf.constant(class_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        y_true = tf.cast(y_true, tf.float32)
        bce = -(y_true * tf.math.log(y_pred) * class_weights +
                (1 - y_true) * tf.math.log(1 - y_pred))
        return tf.reduce_mean(bce)
    return loss


In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dropout, Dense, Embedding, GlobalMaxPooling1D ,Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

classes = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# handle imbalance in the data by use weighted_binary_crossentropy



model = Sequential()

model.add(Embedding(30000, 300))

model.add(Bidirectional(SimpleRNN(64, activation='tanh', return_sequences=True)))

model.add(GlobalMaxPooling1D())

model.add(Dropout(0.4))

model.add(Dense(120, activation='relu'))

model.add(Dropout(0.4))

model.add(Dense(64, activation='relu'))

model.add(Dropout(0.30))

# Multi-label classification
model.add(Dense(6, activation='sigmoid'))


model.compile(
    loss=weighted_binary_crossentropy(class_weights),
    optimizer=tf.keras.optimizers.Adam(1e-3),
    metrics=[tf.keras.metrics.AUC(multi_label=True, name='auc')]
)



model.summary()

# early stoping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# train
history = model.fit(
    train_dataset,
    epochs=5,
    validation_data=val_dataset,
    callbacks=[early_stop]
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
   9/5358 ━━━━━━━━━━━━━━━━━━━━ 1:24 16ms/step - auc: 0.4935 - loss: 0.7265

I0000 00:00:1790989066.553313     133 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5358/5358 ━━━━━━━━━━━━━━━━━━━━ 68s 11ms/step - auc: 0.9785 - loss: 0.1657 - val_auc: 0.9917 - val_loss: 0.0994
Epoch 2/5
5358/5358 ━━━━━━━━━━━━━━━━━━━━ 55s 10ms/step - auc: 0.9949 - loss: 0.0672 - val_auc: 0.9924 - val_loss: 0.0892
Epoch 3/5
5358/5358 ━━━━━━━━━━━━━━━━━━━━ 55s 10ms/step - auc: 0.9963 - loss: 0.0549 - val_auc: 0.9919 - val_loss: 0.0915
Epoch 4/5
5358/5358 ━━━━━━━━━━━━━━━━━━━━ 55s 10ms/step - auc: 0.9972 - loss: 0.0472 - val_auc: 0.9920 - val_loss: 0.0905


In [11]:
from sklearn.metrics import precision_recall_curve, f1_score, classification_report
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Embedding, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

y_val_pred_prob = model.predict(val_dataset)


threshold = 0.5
y_val_pred = (y_val_pred_prob >= threshold).astype(int)


print("Macro F1:   %.4f" % f1_score(y_val, y_val_pred, average='macro', zero_division=0))
print("Micro F1:   %.4f" % f1_score(y_val, y_val_pred, average='micro', zero_division=0))
print("Weighted F1: %.4f" % f1_score(y_val, y_val_pred, average='weighted', zero_division=0))

print("\n" + "=" * 60)
print(classification_report(
    y_val, y_val_pred,
    target_names=label_cols,
    zero_division=0
))

1148/1148 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step
Macro F1:   0.9540
Micro F1:   0.9495
Weighted F1: 0.9493

               precision    recall  f1-score   support

        toxic       0.98      0.90      0.94      9222
 severe_toxic       0.94      0.95      0.95      3272
      obscene       0.96      0.92      0.94      5793
       threat       0.97      0.98      0.98      2526
       insult       0.97      0.92      0.94      6690
identity_hate       0.96      0.98      0.97      3602

    micro avg       0.97      0.93      0.95     31105
    macro avg       0.97      0.94      0.95     31105
 weighted avg       0.97      0.93      0.95     31105
  samples avg       0.45      0.45      0.45     31105



In [12]:
from sklearn.metrics import precision_recall_curve, f1_score, classification_report
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Embedding, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

y_test_pred_prob = model.predict(test_dataset)


threshold = 0.5
y_test_pred = (y_test_pred_prob >= threshold).astype(int)


print("Macro F1:   %.4f" % f1_score(y_test, y_test_pred, average='macro', zero_division=0))
print("Micro F1:   %.4f" % f1_score(y_test, y_test_pred, average='micro', zero_division=0))
print("Weighted F1: %.4f" % f1_score(y_test, y_test_pred, average='weighted', zero_division=0))

print("\n" + "=" * 60)
print(classification_report(
    y_test, y_test_pred,
    target_names=label_cols,
    zero_division=0
))

1148/1148 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
Macro F1:   0.9539
Micro F1:   0.9487
Weighted F1: 0.9485

               precision    recall  f1-score   support

        toxic       0.99      0.90      0.94      9222
 severe_toxic       0.95      0.95      0.95      3302
      obscene       0.96      0.92      0.94      5793
       threat       0.98      0.99      0.98      2528
       insult       0.97      0.92      0.94      6667
identity_hate       0.96      0.98      0.97      3601

    micro avg       0.97      0.93      0.95     31113
    macro avg       0.97      0.94      0.95     31113
 weighted avg       0.97      0.93      0.95     31113
  samples avg       0.44      0.45      0.44     31113

